# 03 — prioritizr optimization (min-shortfall, MGA gap-portfolio)

Run the conservation optimization in **R** on the aligned hand-off stack and write results
back for Python (`04`). Reads the **manifest** contract written by `02`
(`input_data/aligned_stack/manifest.json`) and validates the grid before optimizing.

**Formulation (iteration 1, see `CLAUDE.md` / project memory):**

- **Minimum-shortfall objective** with an **area budget = 30% of the region** (30x30).
  Within that budget it minimizes the weighted shortfall from per-feature targets — the
  best balanced representation achievable in 30%. Targets are *soft*, so no single feature
  or EFG is forced to an overwhelming level.
- **Protected areas locked in** and counted toward the budget.
- **EFG down-weighting** so the 40 EFGs do not swamp the 9 continuous features.
- **Connectivity penalty** (from the transboundary-connectivity surface) for contiguity —
  favors corridors linking PAs over scattered pixels. Off by default (magnitude is
  scale-dependent); calibrate via `CONNECTIVITY_PENALTY` in `config.py`.
- **MGA gap-portfolio** (`add_gap_portfolio`): a suite of structurally diverse
  near-optimal alternatives. **Requires Gurobi** (free academic license).

**Outputs** -> `output_data/iter1_minshortfall30/`: `portfolio.tif` (one alternative per
band), `selection_frequency.tif` (robustness/priority), `portfolio_representation.csv`
(feeds the 04 radar plot), `run_summary.json`.

**Kernel:** select **`R (y2y)`**. Run cell-by-cell; Ethan runs, Claude never executes.

In [ ]:
# ---- Setup: libraries, paths, run parameters (from the manifest) ----------
suppressPackageStartupMessages({
  library(prioritizr)
  library(terra)
  library(jsonlite)
})

PROJ <- normalizePath(getwd())                 # run from the project root
MANIFEST <- file.path(PROJ, "input_data", "aligned_stack", "manifest.json")
stopifnot("manifest.json not found -- run the final cell of 02 first" = file.exists(MANIFEST))

manifest <- jsonlite::read_json(MANIFEST, simplifyVector = TRUE)
params <- manifest$params; grid <- manifest$grid; layers <- manifest$layers

# Run parameters: single source of truth is config.py, carried via the manifest.
SOLVER        <- params$solver                 # "highs" (prototype) or "gurobi" (portfolio)
TIME_LIMIT    <- params$solver_time_limit
BUDGET_PCT    <- params$budget_pct
TARGET_PCT    <- params$target_pct
OPT_GAP       <- params$opt_gap
PORTFOLIO_N   <- params$portfolio_n
PORTFOLIO_GAP <- params$portfolio_gap
CONN_PENALTY  <- params$connectivity_penalty
BOUND_PENALTY <- params$boundary_penalty
RUN_TAG <- params$results_subdir
OUT_DIR <- file.path(PROJ, params$results_dir, RUN_TAG)
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

USE_PORTFOLIO <- identical(SOLVER, "gurobi")   # gap-portfolio only with Gurobi
have_gurobi <- requireNamespace("gurobi", quietly = TRUE)

cat(sprintf("prioritizr %s | terra %s | solver=%s%s\n",
            packageVersion("prioritizr"), packageVersion("terra"), SOLVER,
            if (USE_PORTFOLIO) sprintf(" (gap-portfolio n=%d)", PORTFOLIO_N) else " (single solution)"))
cat(sprintf("budget=%.0f%% target=%.0f%% | opt_gap=%.2f | time_limit=%ss\n",
            100*BUDGET_PCT, 100*TARGET_PCT, OPT_GAP, TIME_LIMIT))
cat(sprintf("connectivity_penalty=%g boundary_penalty=%g\n", CONN_PENALTY, BOUND_PENALTY))
cat(sprintf("outputs -> %s\n", sub(paste0(PROJ, "/"), "", OUT_DIR)))

In [ ]:
# ---- Ingest the hand-off stack + validate the grid (the Python->R contract) ----
abspath <- function(p) file.path(PROJ, p)

is_cont <- layers$role == "feature_continuous"
is_efg  <- layers$role == "feature_efg"
feat_rows <- layers[is_cont | is_efg, ]          # continuous first, then EFG (manifest order)
cost_row  <- layers[layers$role == "cost", ]
pa_row    <- layers[layers$role == "mask_locked_in", ]

features <- terra::rast(abspath(feat_rows$path)); names(features) <- feat_rows$name
cost     <- terra::rast(abspath(cost_row$path));  names(cost) <- "cost"
pa       <- terra::rast(abspath(pa_row$path));    names(pa) <- "pa"

# Validate every layer shares the canonical grid declared in the manifest.
b <- grid$bounds                                 # [left, bottom, right, top]
chk <- function(r, nm, problems) {
  if (terra::ncol(r) != grid$width || terra::nrow(r) != grid$height)
    problems <- c(problems, sprintf("%s: dim %dx%d != %dx%d", nm,
                  terra::ncol(r), terra::nrow(r), grid$width, grid$height))
  e <- unname(as.vector(terra::ext(r)))          # xmin,xmax,ymin,ymax
  if (!isTRUE(all.equal(e, c(b[1], b[3], b[2], b[4]), tolerance = 1e-3)))
    problems <- c(problems, sprintf("%s: extent mismatch", nm))
  problems
}
problems <- character(0)
for (i in seq_len(terra::nlyr(features))) problems <- chk(features[[i]], names(features)[i], problems)
problems <- chk(cost, "cost", problems); problems <- chk(pa, "pa", problems)
if (length(problems)) stop("GRID VALIDATION FAILED:\n  ", paste(problems, collapse = "\n  "))

cat(sprintf("ingested %d features (%d continuous + %d EFG) + cost + PA mask\n",
            terra::nlyr(features), sum(is_cont), sum(is_efg)))
cat(sprintf("grid OK: %d x %d @ %d m, %s\n", grid$width, grid$height, grid$res_m, grid$crs))

In [ ]:
# ---- Planning units, lock-in, and a feasibility check --------------------
# Cost (=1 per cell, NA outside the corridor) defines the planning units (PUs).
n_pu <- terra::global(!is.na(cost), "sum", na.rm = TRUE)[[1]]
total_cost <- terra::global(cost, "sum", na.rm = TRUE)[[1]]   # == n_pu (uniform cost)
BUDGET <- BUDGET_PCT * total_cost                            # area budget in cost units

# Lock in existing PAs: pa==1 within the PU set (terra read the 255 sentinel as NA
# outside the PU). Restrict to PUs so a non-PU cell can never be locked in.
locked <- terra::mask(pa == 1, cost)
n_locked <- terra::global(locked, "sum", na.rm = TRUE)[[1]]

cat(sprintf("planning units: %s cells | budget = %.0f%% = %s cells\n",
            format(n_pu, big.mark=","), 100*BUDGET_PCT, format(round(BUDGET), big.mark=",")))
cat(sprintf("locked-in PAs : %s cells (%.1f%% of region) -- %s\n",
            format(n_locked, big.mark=","), 100*n_locked/n_pu,
            if (n_locked <= BUDGET) "fits within budget" else "EXCEEDS BUDGET"))
if (n_locked > BUDGET)
  stop("locked-in PA area exceeds the budget -> infeasible; relax BUDGET_PCT or lock-in.")

In [ ]:
# ---- Feature weights: keep 40 EFGs from swamping the 9 continuous features ----
# Each continuous feature gets weight 1; the EFG group shares a total weight of 1
# (each EFG = 1/n_efg), so EFGs collectively count like one extra theme, not 40.
# Order matches the `features` stack (continuous first, then EFG).
n_cont <- sum(is_cont); n_efg <- sum(is_efg)
weights <- c(rep(1.0, n_cont), rep(1.0 / n_efg, n_efg))
names(weights) <- names(features)
cat(sprintf("weights: %d continuous @ 1.0 ; %d EFG @ %.4f (EFG group total = 1.0)\n",
            n_cont, n_efg, 1.0 / n_efg))

In [ ]:
# ---- Connectivity penalty (contiguity through high-connectivity terrain) -----
# Connectivity matrix between neighbouring PUs weighted by the oriented
# transboundary-connectivity surface: selecting connected, high-connectivity cells
# (corridors linking PAs) is rewarded over scattered pixels. The penalty magnitude is
# scale-dependent -- inspect these diagnostics, then set CONNECTIVITY_PENALTY in
# config.py (0 = baseline run, no penalty).
conn_layer <- features[["transboundary_connectivity"]]
cm <- prioritizr::connectivity_matrix(cost, conn_layer)
cat(sprintf("connectivity matrix: %s edges | weight range [%.4g, %.4g], mean %.4g\n",
            format(length(cm@x), big.mark=","), min(cm@x), max(cm@x), mean(cm@x)))
cat(sprintf("CONNECTIVITY_PENALTY = %g (%s)\n", CONN_PENALTY,
            if (CONN_PENALTY > 0) "active" else "OFF -- baseline; raise in config.py after inspecting scale"))

In [ ]:
# ---- Build the conservation problem -------------------------------------
# Minimum-shortfall: within the area budget, minimise the weighted shortfall from the
# per-feature targets. Targets are soft; PAs are locked in. SOLVER picks the engine:
# HiGHS returns ONE solution (prototype, no license cap); Gurobi adds the MGA
# gap-portfolio (suite of near-optimal alternatives; needs an unlimited academic license).
p <- problem(cost, features) |>
  add_min_shortfall_objective(budget = BUDGET) |>
  add_relative_targets(TARGET_PCT) |>
  add_feature_weights(weights) |>
  add_locked_in_constraints(locked) |>
  add_binary_decisions()

if (CONN_PENALTY > 0)  p <- p |> add_connectivity_penalties(CONN_PENALTY, data = cm)
if (BOUND_PENALTY > 0) p <- p |> add_boundary_penalties(BOUND_PENALTY)

n_threads <- parallel::detectCores()
if (USE_PORTFOLIO) {
  stopifnot("SOLVER='gurobi' but the gurobi R package is not installed (see requirements-R.txt)" = have_gurobi)
  p <- p |>
    add_gap_portfolio(number_solutions = PORTFOLIO_N, pool_gap = PORTFOLIO_GAP) |>
    add_gurobi_solver(gap = OPT_GAP, time_limit = TIME_LIMIT, threads = n_threads, verbose = TRUE)
} else {
  # Single-solution prototype with HiGHS -- `s` will have one layer.
  p <- p |> add_highs_solver(gap = OPT_GAP, time_limit = TIME_LIMIT, threads = n_threads, verbose = TRUE)
}

print(p)

In [ ]:
# ---- Solve (Ethan runs; heavy) ------------------------------------------
# Full 1 km / ~1.27M-PU MILP. HiGHS prototype = one solution; Gurobi = the portfolio.
# Capped by TIME_LIMIT (returns the best solution so far); raise OPT_GAP for speed.
timing <- system.time(s <- solve(p))
# `s` is a SpatRaster: one near-optimal alternative per layer (1 layer for HiGHS).
n_sol <- terra::nlyr(s)
names(s) <- sprintf("alt_%02d", seq_len(n_sol))
cat(sprintf("solved with %s: %d solution(s) in %.1f s\n", SOLVER, n_sol, timing[["elapsed"]]))

In [ ]:
# ---- Per-alternative summaries + selection-frequency map ----------------
rep_rows <- list(); stat_rows <- list()
for (i in seq_len(n_sol)) {
  sol <- s[[i]]
  fr <- eval_feature_representation_summary(p, sol)   # feature, total_amount, relative_held, ...
  fr$alternative <- names(s)[i]
  rep_rows[[i]] <- fr
  n_sel <- terra::global(sol, "sum", na.rm = TRUE)[[1]]
  n_new <- terra::global(sol & (locked == 0), "sum", na.rm = TRUE)[[1]]
  stat_rows[[i]] <- data.frame(alternative = names(s)[i], n_selected = n_sel,
                               pct_region = 100 * n_sel / n_pu, n_added_beyond_pa = n_new)
}
representation <- do.call(rbind, rep_rows)
stats <- do.call(rbind, stat_rows)

# Selection frequency: how many alternatives select each cell (robustness / priority).
sel_freq <- sum(s); names(sel_freq) <- "selection_frequency"

print(stats)

In [ ]:
# ---- Write outputs: clean hand-off back to Python (04) -------------------
portfolio_tif <- file.path(OUT_DIR, "portfolio.tif")
freq_tif      <- file.path(OUT_DIR, "selection_frequency.tif")
rep_csv       <- file.path(OUT_DIR, "portfolio_representation.csv")
summary_json  <- file.path(OUT_DIR, "run_summary.json")

terra::writeRaster(s, portfolio_tif, overwrite = TRUE, datatype = "INT1U",
                   NAflag = 255, gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
terra::writeRaster(sel_freq, freq_tif, overwrite = TRUE, datatype = "INT1U",
                   NAflag = 255, gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
write.csv(representation, rep_csv, row.names = FALSE)

run_summary <- list(
  run_tag = RUN_TAG, objective = "min_shortfall", params = params,
  n_planning_units = n_pu, budget_cells = round(BUDGET), n_locked_in = n_locked,
  n_alternatives = n_sol, solve_seconds = unname(timing[["elapsed"]]),
  per_alternative = stats,
  versions = list(R = as.character(getRversion()),
                  prioritizr = as.character(packageVersion("prioritizr")),
                  terra = as.character(packageVersion("terra")),
                  gurobi = if (have_gurobi) as.character(packageVersion("gurobi")) else NA))
jsonlite::write_json(run_summary, summary_json, auto_unbox = TRUE, pretty = TRUE, na = "null")

cat("wrote:\n")
for (f in c(portfolio_tif, freq_tif, rep_csv, summary_json))
  cat(sprintf("  %s\n", sub(paste0(PROJ, "/"), "", f)))